# Stage 1 — Car Parts Segmentation (YOLOv8s-seg)

**Project:** car-damage-morocco — Stage 1 of the multi-stage pipeline.

Trains `yolov8s-seg` on the Ultralytics built-in `carparts-seg` dataset (23 classes).

**Hardware:** Kaggle T4 (single GPU). Expected runtime: ~2–3 hours for 100 epochs.

**Outputs (in `/kaggle/working/`):**
- `parts_seg/weights/best.pt` — best checkpoint (use this in `DamageDetector`)
- `parts_seg/weights/last.pt` — last checkpoint
- `parts_seg/results.png` — training curves
- `parts_seg/confusion_matrix.png` — validation confusion matrix
- `parts_seg/predictions/` — sample inference renders

**Class list (23):** back_bumper, back_door, back_glass, back_left_door, back_left_light, back_light, back_right_door, back_right_light, back_window, front_bumper, front_door, front_glass, front_left_door, front_left_light, front_light, front_right_door, front_right_light, front_window, hood, left_mirror, right_mirror, tailgate, trunk, wheel.

## 1. Environment setup

In [ ]:
import sys, subprocess

def pip(*args):
    return subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)

# Uninstall ray — Kaggle's version breaks ultralytics integration on import
pip('uninstall', '-y', 'ray')
pip('install', '-q', '--no-warn-conflicts', 'ultralytics==8.3.40')

# Purge cached ray/ultralytics so the new install is picked up
for mod in list(sys.modules):
    if (mod == 'ray' or mod.startswith('ray.')
        or mod == 'ultralytics' or mod.startswith('ultralytics.')):
        del sys.modules[mod]

import os, time, json, shutil, random
from pathlib import Path

import numpy as np
import torch

# Backward-compat shim — ultralytics 8.3.40 calls np.trapz which became np.trapezoid in numpy 2.0
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid
    print('Applied np.trapz -> np.trapezoid shim for numpy 2.x')

import ultralytics
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

print('Python      :', sys.version.split()[0])
print('NumPy       :', np.__version__)
print('Ultralytics :', ultralytics.__version__)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'GPU required'
print('Datasets dir:', SETTINGS.get('datasets_dir'))


In [ ]:
# Point Ultralytics at /kaggle/working so dataset + checkpoints persist as notebook output
WORK = Path('/kaggle/working')
DATASETS_DIR = WORK / 'datasets'
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

from ultralytics.utils import SETTINGS
SETTINGS.update({'datasets_dir': str(DATASETS_DIR)})
print('datasets_dir ->', SETTINGS['datasets_dir'])

RUN_DIR = WORK / 'parts_seg'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RUN_DIR.exists():
    print('Reusing existing run dir:', RUN_DIR)
else:
    print('Run dir will be created at:', RUN_DIR)

# === Manual dataset download + YAML resolution ===
# The carparts-seg zip extracts directly into /kaggle/working/datasets/ (no carparts-seg/ wrapper).
import urllib.request, zipfile, yaml
from pathlib import Path

DATA_ROOT      = Path('/kaggle/working/datasets')
DATA_YAML_PATH = DATA_ROOT / 'carparts-seg.yaml'
ZIP_URL        = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/carparts-seg.zip'
ZIP_PATH       = Path('/kaggle/working/carparts-seg.zip')

if not DATA_YAML_PATH.exists():
    print(f'Downloading {ZIP_URL}')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f'  got {ZIP_PATH.stat().st_size / 1e6:.1f} MB, extracting...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_ROOT)
    ZIP_PATH.unlink()

assert DATA_YAML_PATH.exists(), f'Still missing {DATA_YAML_PATH}'

with open(DATA_YAML_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(DATA_ROOT)
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print('
Verifying splits:')
for split_key in ['train', 'val', 'test']:
    rel = cfg.get(split_key)
    if not rel:
        continue
    full = (DATA_ROOT / rel).resolve()
    n = len(list(full.glob('*.jpg')) + list(full.glob('*.png')))
    status = f'OK  ({n} images)' if full.exists() and n > 0 else f'MISSING  ({full})'
    print(f'  {split_key:5s} -> {rel:20s} -> {status}')

print(f'
Classes ({len(cfg["names"])}):')
print(cfg['names'])

DATA_YAML = str(DATA_YAML_PATH)


In [ ]:
import ultralytics, yaml
from pathlib import Path

# Built-in dataset config shipped with the ultralytics package
ul_root = Path(ultralytics.__file__).parent
candidate = ul_root / 'cfg' / 'datasets' / 'carparts-seg.yaml'
assert candidate.exists(), f'carparts-seg.yaml not found at {candidate}. Update ultralytics or pass a custom path.'
DATA_YAML = str(candidate)
print('Using dataset YAML:', DATA_YAML)
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print('Classes (nc =', cfg.get('nc', len(cfg.get('names', []))), '):')
print(cfg.get('names'))

## 3. Train
Reasonable starting hyperparameters for T4 + YOLOv8s-seg on a small-to-medium dataset.
If you hit OOM, drop `batch` to 12 or `imgsz` to 512.

In [ ]:
model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=5e-4,
    momentum=0.937,
    # Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5, shear=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.1, close_mosaic=10,
    # Bookkeeping
    project=str(WORK),
    name='parts_seg',
    exist_ok=True,
    patience=30,
    save=True,
    plots=True,
    seed=42,
    device=0,
    workers=2,
    amp=True,
    verbose=True,
)
print('Training finished. Best weights:', RUN_DIR / 'weights' / 'best.pt')

## 4. Validation
Sanity-check mAP / mask metrics on the validation split and dump per-class results.

In [ ]:
best_pt = RUN_DIR / 'weights' / 'best.pt'
assert best_pt.exists(), f'Missing {best_pt}. Did training complete?'

val_model = YOLO(str(best_pt))
metrics = val_model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=16,
    split='val',
    project=str(WORK),
    name='parts_seg_val',
    exist_ok=True,
    plots=True,
    save_json=False,
    device=0,
)

summary = {
    'box_mAP50':    float(metrics.box.map50),
    'box_mAP50-95': float(metrics.box.map),
    'mask_mAP50':   float(metrics.seg.map50),
    'mask_mAP50-95':float(metrics.seg.map),
}
print(json.dumps(summary, indent=2))
with open(WORK / 'parts_seg_val_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

## 5. Quick visual sanity check
Run inference on a handful of validation images and save the rendered predictions.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

val_imgs_dir = Path(cfg['path']) / cfg.get('val', 'images/val') if 'path' in cfg else None
if val_imgs_dir is None or not val_imgs_dir.exists():
    # Fallback: search under the dataset root Ultralytics resolved
    dataset_root = DATASETS_DIR / 'carparts-seg'
    candidates = list(dataset_root.rglob('val/*'))
    val_imgs_dir = candidates[0].parent if candidates else dataset_root
print('Inference source dir:', val_imgs_dir)

sample = sorted(glob.glob(str(val_imgs_dir / '*.*')))[:5]
print('Sample images:', sample)

pred_dir = WORK / 'parts_seg' / 'predictions'
pred_dir.mkdir(parents=True, exist_ok=True)

results = val_model.predict(
    source=sample,
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(WORK / 'parts_seg'),
    name='predictions',
    exist_ok=True,
    device=0,
)

rendered = sorted(glob.glob(str(pred_dir / '*.jpg')) + glob.glob(str(pred_dir / '*.png')))[:5]
for p in rendered:
    plt.figure(figsize=(8, 6))
    plt.imshow(Image.open(p))
    plt.axis('off')
    plt.title(Path(p).name)
    plt.show()

## 6. Persist deliverables
Make sure `best.pt` and metrics are easy to grab from the Kaggle output tab.

In [ ]:
import shutil

out = WORK / 'stage1_deliverables'
out.mkdir(exist_ok=True)

shutil.copy(RUN_DIR / 'weights' / 'best.pt', out / 'parts_seg_best.pt')
for fname in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'BoxPR_curve.png', 'MaskPR_curve.png']:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, out / fname)

if (WORK / 'parts_seg_val_summary.json').exists():
    shutil.copy(WORK / 'parts_seg_val_summary.json', out / 'val_summary.json')

# Also save class-name list as JSON so downstream code (DamageDetector) can import it
with open(out / 'parts_classes.json', 'w') as f:
    json.dump(cfg['names'], f, indent=2)

print('Deliverables in', out)
for p in sorted(out.iterdir()):
    print(' -', p.name, p.stat().st_size, 'bytes')

## 7. Next steps (local)
1. Download `stage1_deliverables/parts_seg_best.pt` from the Kaggle output panel.
2. Place it at `models/parts_seg/best.pt` in the `car-damage-morocco` repo.
3. Wire it into `DamageDetector` by loading with `YOLO('models/parts_seg/best.pt')`.
4. Use `parts_classes.json` as the canonical class→id map for the fusion + pricing lookup.